# Document Data Extraction

In [1]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [2]:
load_dotenv()
SARVAM_API_KEY = os.getenv('SARVAM_API_KEY')

assert SARVAM_API_KEY is not None, 'Could NOT load SARVAM_API_KEY from .env'    

In [3]:
from sarvamai import SarvamAI
# from sarvamai.core.api_error import ApiError

In [4]:
client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
)

## Extract Data from Document

In [5]:
def load_document(f_name: str):
    with open(f_name, 'rb') as f_in:
        return f_in.read()

In [6]:
data = load_document('./data/pdfs/letter_to_bank.pdf')

### Define data elements that need to be extracted

In [14]:
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "Full Name of sender"},
        "address": {"type": "string", "description": "Address of sender"},
        "phone_number": {"type": "string", "description": "Mobile number of sender"},
        "request_summary":  {"type": "string", "description": "Summary of Request"},
        "account_number":   {"type": "string", "description": "Account Number of sender (if present)"},
        "client_id":   {"type": "number", "description": "Client Id of sender (if present)"},
    },
}

### Submit Job

In [15]:
import json

In [16]:
job = client.doc_ai.extract(
    file=[('output', data, 'application/pdf')],
    language='en-IN',
    schema=json.dumps(schema),
    output_format='json'
)

In [17]:
job.job_id

'01a176c1-089c-741a-98dd-ab678a924c82'

### Check Job Status

In [18]:
status = client.doc_ai.get_status(job_id=job.job_id)
json.loads(status.json())

{'job_id': '01a176c1-089c-741a-98dd-ab678a924c82',
 'status': 'pending',
 'pipeline': 'extract',
 'usage': {'pages_total': 0,
  'pages_processed': 0,
  'pages_succeeded': 0,
  'pages_failed': 0},
 'created_at': '2026-10-26T05:48:44Z',
 'updated_at': '2026-10-26T05:48:44Z',
 '$schema': 'https://api.sarvam.ai/schemas/PublicJobBody.json',
 'documents': [{'filename': 'output',
   'status': 'accepted',
   'language': 'en-IN'}]}

### Get Extracted Data

In [19]:
extracted_data = client.doc_ai.get_results(job_id=job.job_id)
extracted_data.result

{'name': 'Amit Sharma',
 'address': 'Chennai - 600001',
 'phone_number': '9840012345',
 'request_summary': 'Closure of Account',
 'account_number': '001002',
 'client_id': None}